# Modelo de classificação

O notebook recorta cada imagem no centro para 300 × 300, calcula a SVD até k_modelo e concatena B e C em um vetor. As dez camadas ocultas têm piso de dois terços da largura anterior e ativação sigmoide; a saída tem um neurônio sigmoide. Cat corresponde a 0 e Dog a 1. A função de perda é MSE.

As configurações editáveis estão na primeira célula de código: k_modelo, escala_cinza, taxa_aprendizado, epocas, tamanho_batch, fracao_treino, usar_todos_dados e N. Se usar_todos_dados=False, N deve ser par e seleciona N/2 imagens de cada classe; semente_amostragem controla a seleção. As imagens vêm de dataset/PetImages/Cat e Dog; a divisão aleatória deixa 90% para treino e 10% para avaliação. A primeira passagem calcula e guarda os vetores SVD em datasets_processados/modelo/300x300/k_<k>/<modo>. A última célula treina e plota o MSE de treino e avaliação. Execute as células em ordem.

A saída 0,5 é o ponto médio da sigmoide; ela não é, por si só, uma probabilidade calibrada. A avaliação de 10% não é um teste final reservado.

## Obter as imagens em outro computador

Execute esta célula antes de criar o conjunto de dados. Na primeira vez, instale kagglehub no ambiente do notebook com %pip install kagglehub e reinicie o kernel. O download usa o espelho do conjunto Microsoft Cats vs. Dogs no Kaggle. Se dataset/PetImages/Cat e Dog já contêm imagens, a célula não baixa nada. É necessário ter acesso ao Kaggle.


In [ ]:
from pathlib import Path
import shutil
import zipfile

diretorio_dados = Path("dataset/PetImages")
extensoes_imagem = {".jpg", ".jpeg", ".png"}

def tem_imagens(pasta):
    return pasta.is_dir() and any(
        arquivo.is_file() and arquivo.suffix.lower() in extensoes_imagem
        for arquivo in pasta.iterdir()
    )

if tem_imagens(diretorio_dados / "Cat") and tem_imagens(diretorio_dados / "Dog"):
    print(f"Dataset já disponível em {diretorio_dados}; download dispensado.")
else:
    if diretorio_dados.exists():
        raise RuntimeError(
            f"{diretorio_dados} já existe, mas não contém as duas classes completas. "
            "Revise essa pasta antes de baixar novamente."
        )
    try:
        import kagglehub
    except ImportError as erro:
        raise ImportError(
            "Instale kagglehub com %pip install kagglehub, reinicie o kernel e execute novamente."
        ) from erro

    pasta_download = Path("dataset/_download_kaggle")
    pasta_download.mkdir(parents=True, exist_ok=True)
    kagglehub.dataset_download(
        "shaunthesheep/microsoft-catsvsdogs-dataset",
        output_dir=str(pasta_download),
    )

    candidatos = [
        pasta for pasta in pasta_download.rglob("PetImages")
        if tem_imagens(pasta / "Cat") and tem_imagens(pasta / "Dog")
    ]
    if candidatos:
        shutil.move(str(candidatos[0]), str(diretorio_dados))
    else:
        # Algumas versões do dataset vêm como arquivo ZIP.
        for arquivo_zip in pasta_download.rglob("*.zip"):
            with zipfile.ZipFile(arquivo_zip) as zip_ref:
                membros = [
                    membro for membro in zip_ref.infolist()
                    if not membro.is_dir() and "PetImages" in Path(membro.filename).parts
                    and Path(membro.filename).suffix.lower() in extensoes_imagem
                ]
                classes = {
                    Path(membro.filename).parts[
                        Path(membro.filename).parts.index("PetImages") + 1
                    ] for membro in membros
                    if len(Path(membro.filename).parts) > Path(membro.filename).parts.index("PetImages") + 1
                }
                if not {"Cat", "Dog"}.issubset(classes):
                    continue
                for membro in membros:
                    partes = Path(membro.filename).parts
                    indice = partes.index("PetImages")
                    if len(partes) != indice + 3 or partes[indice + 1] not in ("Cat", "Dog"):
                        continue
                    destino = diretorio_dados / partes[indice + 1] / partes[indice + 2]
                    destino.parent.mkdir(parents=True, exist_ok=True)
                    with zip_ref.open(membro) as origem, destino.open("wb") as saida:
                        shutil.copyfileobj(origem, saida)
                break

    if not (tem_imagens(diretorio_dados / "Cat") and tem_imagens(diretorio_dados / "Dog")):
        raise RuntimeError(
            "O download terminou, mas não encontrei PetImages/Cat e PetImages/Dog. "
            f"Confira os arquivos em {pasta_download}."
        )
    print(f"Imagens prontas em {diretorio_dados}.")


In [ ]:
import numpy as np
from PIL import Image
from scipy.sparse.linalg import svds, ArpackError, ArpackNoConvergence
import warnings
from pathlib import Path


# Configurações do modelo e do treinamento
k_modelo = 20
escala_cinza = True  # False: manter RGB
taxa_aprendizado = 0.1
epocas = 1000
tamanho_batch = 50
fracao_treino = 0.90
usar_todos_dados = True  # False: usar apenas N imagens no total
N = 1000  # quando usar_todos_dados=False: metade Cat, metade Dog
semente_amostragem = 42

TAMANHO_ALVO = (300, 300)  # largura, altura; recorte quadrado

def preparar_imagem(A, escala_cinza=True):
    """Recorta o centro em um quadrado e redimensiona para 300 x 300 pixels."""
    if not isinstance(escala_cinza, (bool, np.bool_)):
        raise TypeError("escala_cinza deve ser True ou False.")
    if isinstance(A, Image.Image):
        imagem = A
    else:
        matriz = np.asarray(A)
        if matriz.ndim not in (2, 3) or (matriz.ndim == 3 and matriz.shape[2] != 3):
            raise ValueError("A deve ter formato (altura, largura) ou (altura, largura, 3).")
        if matriz.ndim == 3 and matriz.dtype != np.uint8:
            raise TypeError("Uma matriz RGB deve conter pixels uint8.")
        imagem = Image.fromarray(matriz)
    imagem = imagem.convert("L" if escala_cinza else "RGB")
    largura, altura = imagem.size
    lado_recorte = min(largura, altura)
    esquerda = (largura - lado_recorte) // 2
    topo = (altura - lado_recorte) // 2
    imagem = imagem.crop((esquerda, topo, esquerda + lado_recorte, topo + lado_recorte))
    if imagem.size != TAMANHO_ALVO:
        imagem = imagem.resize(TAMANHO_ALVO)
    return np.asarray(imagem, dtype=float)

def fatorar_canal(canal, k):
    canal = np.asarray(canal, dtype=float)
    if canal.ndim != 2:
        raise ValueError("Cada canal deve ser uma matriz bidimensional.")
    if not isinstance(k, (int, np.integer)) or not 1 <= k < min(canal.shape):
        raise ValueError(f"k deve ser inteiro entre 1 e {min(canal.shape) - 1}.")
    try:
        U_k, sigma_k, V_k_T = svds(canal, k=k, which="LM")
    except (ArpackError, ArpackNoConvergence) as erro:
        warnings.warn(
            f"ARPACK falhou para canal {canal.shape}; usando SVD densa e mantendo k={k}. "
            f"Detalhe: {erro}",
            RuntimeWarning,
            stacklevel=2,
        )
        U_completo, sigma_completo, V_completo_T = np.linalg.svd(
            canal, full_matrices=False
        )
        U_k = U_completo[:, :k]
        sigma_k = sigma_completo[:k]
        V_k_T = V_completo_T[:k, :]
    ordem_decrescente = np.argsort(sigma_k)[::-1]
    U_k = U_k[:, ordem_decrescente]
    sigma_k = sigma_k[ordem_decrescente]
    V_k_T = V_k_T[ordem_decrescente, :]
    B_k = U_k
    C_k = sigma_k[:, None] * V_k_T
    return B_k, C_k


def transformar(A, k, escala_cinza=True):
    """Retorna (B_k, C_k) em cinza ou seis fatores na ordem RGB."""
    imagem = preparar_imagem(A, escala_cinza=escala_cinza)
    if escala_cinza:
        return fatorar_canal(imagem, k)
    B_R, C_R = fatorar_canal(imagem[:, :, 0], k)
    B_G, C_G = fatorar_canal(imagem[:, :, 1], k)
    B_B, C_B = fatorar_canal(imagem[:, :, 2], k)
    return B_R, C_R, B_G, C_G, B_B, C_B


def montar_vetor_entrada(A):
    """Concatena os fatores definidos por k_modelo em um vetor float32."""
    if not isinstance(k_modelo, int) or not 1 <= k_modelo < 300:
        raise ValueError("Defina k_modelo como inteiro entre 1 e 299.")
    fatores = transformar(A, k_modelo, escala_cinza=escala_cinza)
    return np.concatenate([fator.ravel() for fator in fatores]).astype(np.float32)

## Preparo da imagem e entrada

Cada imagem é recortada no maior quadrado central e redimensionada para 300 × 300 pixels, sem esticar. O redimensionamento usa o padrão do Pillow. O valor de k_modelo e o booleano escala_cinza ficam no início da primeira célula de código.

Em cinza, transformar retorna B_k e C_k, ambos com 300k elementos. montar_vetor_entrada concatena os fatores e produz 600k números. Em RGB, a função retorna B_R, C_R, B_G, C_G, B_B e C_B, produzindo 1800k números após concatenação.

A rede recebe um lote de vetores com formato (quantidade_de_imagens, entrada_dim).

## PyTorch

A conversa incluiu a instalação com `%pip install --user torch torchvision` para um ambiente compartilhado. Execute essa instalação separadamente somente se for necessária no ambiente em que abrir o notebook.


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
import time
from torch.utils.data import Dataset, DataLoader, random_split

In [ ]:
class RedeNeural(nn.Module):
    def __init__(self, entrada_dim):
        super().__init__()
        if not isinstance(entrada_dim, int) or entrada_dim < 1:
            raise ValueError("entrada_dim deve ser um inteiro positivo.")

        self.camadas_ocultas = nn.ModuleList()
        largura_atual = entrada_dim
        for _ in range(10):
            largura_seguinte = (2 * largura_atual) // 3
            if largura_seguinte < 1:
                raise ValueError("A regra de 2/3 produziu uma camada vazia.")
            self.camadas_ocultas.append(nn.Linear(largura_atual, largura_seguinte))
            largura_atual = largura_seguinte

        self.saida = nn.Linear(largura_atual, 1)
        self.sigmoide = nn.Sigmoid()

    def forward(self, x):
        for camada in self.camadas_ocultas:
            x = self.sigmoide(camada(x))
        return self.sigmoide(self.saida(x))

In [ ]:
# Configure k_modelo e escala_cinza na primeira célula de código.
if not isinstance(k_modelo, int) or not 1 <= k_modelo < 300:
    raise ValueError("Defina k_modelo como inteiro entre 1 e 299 na primeira célula.")
entrada_dim = 2 * 300 * k_modelo * (1 if escala_cinza else 3)
modelo = RedeNeural(entrada_dim)



In [ ]:
class ConjuntoImagensSVD(Dataset):
    """Lê Cat/Dog e armazena cada vetor SVD depois do primeiro cálculo."""

    def __init__(self, diretorio, usar_todos_dados=True, N=1000, semente_amostragem=42):
        if not isinstance(usar_todos_dados, bool):
            raise TypeError("usar_todos_dados deve ser True ou False.")
        if not usar_todos_dados and (not isinstance(N, int) or isinstance(N, bool) or N < 2 or N % 2):
            raise ValueError("N deve ser um inteiro par e maior ou igual a 2.")
        self.amostras = []
        invalidos = []
        for classe, rotulo in (("Cat", 0.0), ("Dog", 1.0)):
            pasta = diretorio / classe
            if not pasta.is_dir():
                raise FileNotFoundError(f"Pasta não encontrada: {pasta}")
            arquivos = sorted(
                caminho for caminho in pasta.iterdir()
                if caminho.is_file() and caminho.suffix.lower() in (".jpg", ".jpeg", ".png")
            )
            for caminho in arquivos:
                try:
                    with Image.open(caminho) as imagem:
                        imagem.verify()
                except (OSError, ValueError) as erro:
                    invalidos.append((caminho, str(erro)))
                else:
                    self.amostras.append((classe, caminho, rotulo))
        if invalidos:
            print(f"{len(invalidos)} imagem(ns) ilegível(is) fora do conjunto:")
            for caminho, erro in invalidos:
                print(f"  {caminho}: {erro}")
        if not self.amostras:
            raise ValueError("Nenhuma imagem legível encontrada.")
        if not usar_todos_dados:
            gerador = np.random.default_rng(semente_amostragem)
            selecionadas = []
            for classe in ("Cat", "Dog"):
                candidatas = [amostra for amostra in self.amostras if amostra[0] == classe]
                quantidade = N // 2
                if len(candidatas) < quantidade:
                    raise ValueError(
                        f"N={N} exige {quantidade} imagens de {classe}, "
                        f"mas apenas {len(candidatas)} são legíveis."
                    )
                indices = gerador.choice(len(candidatas), size=quantidade, replace=False)
                selecionadas.extend(candidatas[int(indice)] for indice in indices)
            self.amostras = selecionadas
            print(f"Amostra selecionada: {N // 2} gatos e {N // 2} cachorros.")
        modo = "cinza" if escala_cinza else "rgb"
        self.cache = Path("datasets_processados") / "modelo" / "300x300" / f"k_{k_modelo}" / modo
        self.cache.mkdir(parents=True, exist_ok=True)

    def __len__(self):
        return len(self.amostras)

    def __getitem__(self, indice):
        classe, caminho, rotulo = self.amostras[indice]
        cache_classe = self.cache / classe
        cache_classe.mkdir(exist_ok=True)
        arquivo_cache = cache_classe / f"{caminho.name}.npy"
        vetor = None
        if arquivo_cache.exists() and arquivo_cache.stat().st_mtime >= caminho.stat().st_mtime:
            try:
                vetor = np.load(arquivo_cache, allow_pickle=False)
            except (OSError, ValueError):
                pass
        if vetor is None:
            with Image.open(caminho) as imagem:
                vetor = montar_vetor_entrada(imagem)
            np.save(arquivo_cache, vetor)
        if vetor.shape != (entrada_dim,):
            raise ValueError(f"Entrada com dimensão inesperada em {caminho}: {vetor.shape}")
        X = torch.from_numpy(vetor.astype(np.float32, copy=False))
        y = torch.tensor([rotulo], dtype=torch.float32)
        return X, y


if not isinstance(tamanho_batch, int) or tamanho_batch < 1:
    raise ValueError("tamanho_batch deve ser um inteiro positivo.")
if not 0 < fracao_treino < 1:
    raise ValueError("fracao_treino deve estar entre 0 e 1.")

diretorio_dados = Path("dataset/PetImages")
dados = ConjuntoImagensSVD(diretorio_dados, usar_todos_dados, N, semente_amostragem)
n_treino = int(fracao_treino * len(dados))
n_avaliacao = len(dados) - n_treino
if n_treino < 1 or n_avaliacao < 1:
    raise ValueError("A divisão deve deixar pelo menos uma imagem em cada conjunto.")
dados_treino, dados_avaliacao = random_split(dados, [n_treino, n_avaliacao])
loader_treino = DataLoader(dados_treino, batch_size=tamanho_batch)
loader_avaliacao = DataLoader(dados_avaliacao, batch_size=tamanho_batch)
print(f"Total: {len(dados)} | treino: {n_treino} | avaliação: {n_avaliacao}")

In [ ]:
# O cache evita recalcular a SVD das imagens em cada época.
# Esta primeira passagem pode demorar; os arquivos ficam em datasets_processados/modelo/300x300/k_<k>/<modo>/.
for indice in range(len(dados)):
    dados[indice]
    if (indice + 1) % 100 == 0 or indice + 1 == len(dados):
        print(f"Vetores SVD preparados: {indice + 1}/{len(dados)}")

In [ ]:
def mse_no_conjunto(modelo, dataloader, criterio):
    modelo.eval()
    perda_total = 0.0
    total_amostras = 0
    with torch.no_grad():
        for X, y in dataloader:
            previsao = modelo(X)
            if y.shape != previsao.shape:
                raise ValueError("y deve ter formato (tamanho_do_lote, 1).")
            y = y.to(dtype=previsao.dtype, device=previsao.device)
            quantidade = X.shape[0]
            perda_total += criterio(previsao, y).item() * quantidade
            total_amostras += quantidade
    return perda_total / total_amostras


def treinar(modelo, loader_treino, loader_avaliacao, epocas, taxa_aprendizado):
    criterio = nn.MSELoss()
    otimizador = optim.Adam(modelo.parameters(), lr=taxa_aprendizado)
    historico = {"treino": [], "avaliacao": []}
    inicio_treinamento = time.perf_counter()
    for epoca in range(epocas):
        modelo.train()
        perda_total = 0.0
        total_amostras = 0
        for X, y in loader_treino:
            otimizador.zero_grad()
            previsao = modelo(X)
            if y.shape != previsao.shape:
                raise ValueError("y deve ter formato (tamanho_do_lote, 1).")
            y = y.to(dtype=previsao.dtype, device=previsao.device)
            perda = criterio(previsao, y)
            perda.backward()
            otimizador.step()
            quantidade = X.shape[0]
            perda_total += perda.item() * quantidade
            total_amostras += quantidade

        mse_treino = perda_total / total_amostras
        mse_avaliacao = mse_no_conjunto(modelo, loader_avaliacao, criterio)
        historico["treino"].append(mse_treino)
        historico["avaliacao"].append(mse_avaliacao)
        tempo_decorrido = time.perf_counter() - inicio_treinamento
        segundos_restantes = tempo_decorrido / (epoca + 1) * (epocas - epoca - 1)
        horas, resto = divmod(int(segundos_restantes), 3600)
        minutos, segundos = divmod(resto, 60)
        tempo_restante = f"{horas:02d}:{minutos:02d}:{segundos:02d}"
        print(
            f"Época {epoca + 1}/{epocas} | "
            f"MSE treino = {mse_treino:.6f} | "
            f"MSE avaliação = {mse_avaliacao:.6f} | "
            f"restante estimado = {tempo_restante}"
        )
    return historico

In [ ]:
def plotar_treinamento(historico):
    epocas_plotadas = range(1, len(historico["treino"]) + 1)
    plt.figure(figsize=(10, 5))
    plt.plot(epocas_plotadas, historico["treino"], label="Treino")
    plt.plot(epocas_plotadas, historico["avaliacao"], label="Avaliação")
    plt.xlabel("Época")
    plt.ylabel("MSE")
    plt.title("Evolução do MSE")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.show()

In [ ]:
historico = treinar(
    modelo,
    loader_treino,
    loader_avaliacao,
    epocas,
    taxa_aprendizado,
)
plotar_treinamento(historico)